# Madhya Pradesh Weekly District Brief — Compute Pipeline
File-based ingestion (CSV / PDF exports from agmarknet.gov.in's Daily Price and Arrival Report),
since the live scrape/API paths are blocked in this environment (see `execution_plan.docx` and
`final_implementation_plan_1.docx` for the full context and decision trail).

**Scope**: Madhya Pradesh only. Every number in the fact sheet is either a raw field from an input
file or a value computed directly from those raw fields in this notebook — nothing is invented,
estimated, or filled in by a language model. This notebook covers Day 1 (ingest) + Day 2 (compute)
of the implementation plan. Narration, translation, and the numeric gate are separate, later steps
that consume this notebook's output — not part of this notebook.

**How to use this notebook**
1. Drop CSV exports into `data/csv/` and PDF exports into `data/pdf/` (see folder-creation cell below).
2. Run all cells top to bottom.
3. Per-district fact sheets are written to `fact_sheets/<district>.json`.

Column names are NOT assumed to be clean or consistent — Agmarknet exports (and PDF table
extraction especially) are known to vary in header spelling/casing between reports. The
`FIELD_ALIASES` map below is the single place to extend when a new column-name variant shows up;
extend it rather than special-casing a loader.

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timedelta

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 0. Paths & config

`WEEK_END` anchors the reporting week (defaults to the latest date seen in the data once loaded —
overridable below). The pipeline always needs the target week's files AND the prior week's files
present in `data/csv` / `data/pdf` to compute week-on-week price change; if only one week is
supplied, that comparison is left explicitly `None` rather than guessed.

In [ ]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
CSV_DIR = DATA_DIR / "csv"
PDF_DIR = DATA_DIR / "pdf"
FACT_SHEET_DIR = BASE_DIR / "fact_sheets"

for d in (CSV_DIR, PDF_DIR, FACT_SHEET_DIR):
    d.mkdir(parents=True, exist_ok=True)

STATE_NAME = "Madhya Pradesh"

# Set explicitly once real data is loaded, or leave None to auto-detect from the data
# (max arrival_date found = week_end, week_start = week_end - 6 days).
WEEK_END_OVERRIDE = None  # e.g. "2026-07-23"

print(f"CSV inputs expected in: {CSV_DIR}")
print(f"PDF inputs expected in: {PDF_DIR}")
print(f"Fact sheets written to: {FACT_SHEET_DIR}")
print(f"CSV files found now:  {sorted(p.name for p in CSV_DIR.glob('*.csv'))}")
print(f"PDF files found now:  {sorted(p.name for p in PDF_DIR.glob('*.pdf'))}")

## 1. Field aliases

Canonical schema this pipeline works in, and every raw column-name variant seen so far that maps
to it. `arrival_qty` (tonnes) is included per the corrected plan (v3.0) — Agmarknet's Daily Price
and Arrival Report carries it alongside price. Extend this dict — don't add per-file special
casing in the loaders below — the moment a new file uses a header not listed here, the loader
will flag it as an *unmapped column* (see the `unmapped_columns` warning in section 2) rather than
silently dropping it.

In [ ]:
FIELD_ALIASES = {
    "state": ["state", "State", "State/UT", "state_name", "State Name"],
    "district": ["district", "District", "district_name", "District Name"],
    "market": ["market", "Market", "market_name", "Market Name"],
    "commodity": ["commodity", "Commodity", "cmdt_name", "Commodity Name"],
    "commodity_group": ["commodity_group", "Commodity Group", "cmdt_grp_name", "Group"],
    "variety": ["variety", "Variety", "variety_name"],
    "grade": ["grade", "Grade", "grade_name"],
    "arrival_date": [
        "arrival_date", "Arrival_Date", "Reported Date", "reported_date",
        "Price Date", "Arrival Date", "Date",
    ],
    "min_price": ["min_price", "Min_Price", "Min Price", "Minimum Price (Rs./Quintal)"],
    "max_price": ["max_price", "Max_Price", "Max Price", "Maximum Price (Rs./Quintal)"],
    "modal_price": [
        "modal_price", "Modal_Price", "Modal Price", "model_price",
        "Modal Price (Rs./Quintal)",
    ],
    "price_unit": ["price_unit", "unit_name_price", "Price Unit"],
    "arrival_qty": [
        "arrival_qty", "Arrivals (Tonnes)", "Arrival Quantity", "arrival_quantity",
        "Arrivals", "Arrival Qty",
    ],
    "arrival_unit": ["arrival_unit", "unit_name_arrival", "Arrival Unit"],
}

CANONICAL_COLUMNS = list(FIELD_ALIASES.keys())

# Reverse lookup: raw column name (lowercased, stripped) -> canonical name
_ALIAS_LOOKUP = {
    alias.strip().lower(): canonical
    for canonical, aliases in FIELD_ALIASES.items()
    for alias in aliases
}


def map_columns(df: pd.DataFrame, source_label: str):
    """Rename df's columns to the canonical schema using FIELD_ALIASES.
    Returns (renamed_df, unmapped_columns) — unmapped columns are NOT dropped,
    they are kept under their original name so nothing collected by the source
    is silently lost; they are just not part of the canonical schema yet.
    """
    rename_map = {}
    unmapped = []
    for col in df.columns:
        key = str(col).strip().lower()
        if key in _ALIAS_LOOKUP:
            rename_map[col] = _ALIAS_LOOKUP[key]
        else:
            unmapped.append(col)
    renamed = df.rename(columns=rename_map)
    if unmapped:
        print(f"[{source_label}] unmapped columns (kept as-is, not yet in FIELD_ALIASES): {unmapped}")
    return renamed, unmapped

## 2. CSV ingestion

Reads every `*.csv` in `data/csv/`, maps columns to the canonical schema, and tags each row with
its source file (useful for tracing a bad number back to the exact export it came from).

In [ ]:
def load_csv_dir(csv_dir: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(csv_dir.glob("*.csv")):
        try:
            raw = pd.read_csv(path, dtype=str)
        except Exception as exc:
            print(f"[CSV] FAILED to read {path.name}: {exc}")
            continue
        mapped, _ = map_columns(raw, path.name)
        mapped["_source_file"] = path.name
        frames.append(mapped)
        print(f"[CSV] loaded {path.name}: {len(mapped)} rows")
    if not frames:
        return pd.DataFrame(columns=CANONICAL_COLUMNS + ["_source_file"])
    return pd.concat(frames, ignore_index=True, sort=False)


csv_records = load_csv_dir(CSV_DIR)
print(f"\nTotal CSV rows loaded: {len(csv_records)}")
csv_records.head()

## 3. PDF ingestion

Agmarknet PDF exports are typically one table per page. `pdfplumber` extracts each page's table
verbatim; rows that don't extract cleanly (e.g. a merged/malformed cell) are collected in
`pdf_parse_errors` rather than silently skipped, so a parsing gap is visible instead of just
missing from the output.

In [ ]:
import pdfplumber


def load_pdf_dir(pdf_dir: Path):
    frames = []
    parse_errors = []
    for path in sorted(pdf_dir.glob("*.pdf")):
        try:
            with pdfplumber.open(path) as pdf:
                page_tables = []
                for page_num, page in enumerate(pdf.pages, start=1):
                    table = page.extract_table()
                    if not table or len(table) < 2:
                        parse_errors.append((path.name, page_num, "no table extracted"))
                        continue
                    header, *rows = table
                    try:
                        df = pd.DataFrame(rows, columns=header)
                    except Exception as exc:
                        parse_errors.append((path.name, page_num, f"header/row mismatch: {exc}"))
                        continue
                    page_tables.append(df)
                if not page_tables:
                    print(f"[PDF] {path.name}: no tables extracted on any page")
                    continue
                combined = pd.concat(page_tables, ignore_index=True, sort=False)
        except Exception as exc:
            parse_errors.append((path.name, None, f"file open/parse failed: {exc}"))
            print(f"[PDF] FAILED to read {path.name}: {exc}")
            continue
        mapped, _ = map_columns(combined, path.name)
        mapped["_source_file"] = path.name
        frames.append(mapped)
        print(f"[PDF] loaded {path.name}: {len(mapped)} rows")
    if not frames:
        empty = pd.DataFrame(columns=CANONICAL_COLUMNS + ["_source_file"])
    else:
        empty = pd.concat(frames, ignore_index=True, sort=False)
    return empty, parse_errors


pdf_records, pdf_parse_errors = load_pdf_dir(PDF_DIR)
print(f"\nTotal PDF rows loaded: {len(pdf_records)}")
if pdf_parse_errors:
    print(f"PDF parse errors ({len(pdf_parse_errors)}) — pages that did NOT make it into pdf_records:")
    for fname, page_num, reason in pdf_parse_errors:
        print(f"  - {fname} (page {page_num}): {reason}")
pdf_records.head()

## 4. Combine, clean, normalize types

Merges CSV + PDF rows into one raw record set, then:
- parses `arrival_date` (tries multiple formats, since CSV vs PDF exports have used different ones)
- converts price/quantity fields to floats — a missing/unparseable value stays `None`, it is never
  coerced to 0, so downstream aggregation can't mistake "no data" for "zero"
- restricts to Madhya Pradesh rows (defensive — in case an export contains other states)

In [ ]:
DATE_FORMATS = ["%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d", "%d %b %Y", "%d-%b-%Y"]


def parse_date(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    value = str(value).strip()
    if not value or value.upper() in ("NA", "N/A"):
        return None
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            continue
    return None


def to_float(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = str(value).replace(",", "").strip()
    if text == "" or text.upper() in ("NA", "N/A"):
        return None
    try:
        return float(text)
    except ValueError:
        return None


raw = pd.concat([csv_records, pdf_records], ignore_index=True, sort=False)
for col in CANONICAL_COLUMNS:
    if col not in raw.columns:
        raw[col] = None

parse_errors = []
raw["arrival_date_parsed"] = raw["arrival_date"].apply(parse_date)
bad_dates = raw[raw["arrival_date"].notna() & raw["arrival_date_parsed"].isna()]
for idx, row in bad_dates.iterrows():
    parse_errors.append((row.get("_source_file"), idx, "unparseable arrival_date", row.get("arrival_date")))

for col in ("min_price", "max_price", "modal_price", "arrival_qty"):
    raw[col] = raw[col].apply(to_float)

def clean_str_col(series):
    # NaN must stay missing, not become the literal string "nan" (astype(str)
    # on a NaN silently produces "nan", which would then get grouped/ranked
    # as a fake real value downstream).
    return series.where(series.notna(), None).apply(lambda v: str(v).strip() if v is not None else None)


raw["district"] = clean_str_col(raw["district"])
raw["market"] = clean_str_col(raw["market"])
raw["commodity"] = clean_str_col(raw["commodity"])

if "state" in raw.columns and raw["state"].notna().any():
    before = len(raw)
    raw = raw[raw["state"].astype(str).str.strip().str.lower() == STATE_NAME.lower()]
    print(f"Filtered to state={STATE_NAME!r}: {before} -> {len(raw)} rows")

print(f"Total combined rows: {len(raw)}")
if parse_errors:
    print(f"{len(parse_errors)} rows had an unparseable arrival_date — excluded from date-window aggregation:")
    for src, idx, reason, val in parse_errors[:20]:
        print(f"  - source={src} row={idx}: {reason} ({val!r})")

raw.head()

## 5. Reporting week window

`week_end` defaults to the latest parsed date in the data (override with `WEEK_END_OVERRIDE`
above). `week_start` is `week_end - 6 days`. `prior_week_start`/`prior_week_end` is the
immediately preceding 7-day window, used for week-on-week price comparison in section 8 — only
populated if the data actually contains rows in that window.

In [ ]:
if WEEK_END_OVERRIDE:
    week_end = datetime.strptime(WEEK_END_OVERRIDE, "%Y-%m-%d").date()
else:
    valid_dates = raw["arrival_date_parsed"].dropna()
    week_end = valid_dates.max() if not valid_dates.empty else None

if week_end is None:
    print("No valid dates found yet — load data into data/csv or data/pdf and re-run from section 2.")
else:
    week_start = week_end - timedelta(days=6)
    prior_week_end = week_start - timedelta(days=1)
    prior_week_start = prior_week_end - timedelta(days=6)
    print(f"Current week: {week_start} to {week_end}")
    print(f"Prior week:   {prior_week_start} to {prior_week_end}")

    current_week_df = raw[raw["arrival_date_parsed"].between(week_start, week_end)]
    prior_week_df = raw[raw["arrival_date_parsed"].between(prior_week_start, prior_week_end)]
    print(f"Rows in current week: {len(current_week_df)}")
    print(f"Rows in prior week:   {len(prior_week_df)}")

## 6. Market roster & submission-status table

Per the plan's Section 3/6 requirement: every market must appear for every day in the window,
distinguishing three states explicitly —
- **submitted-with-trade**: at least one commodity record with a modal price that day
- **submitted-no-transaction**: a record exists for that market/day but with no usable price
  (a genuine "nil transaction" report, not a missing one)
- **not-submitted**: no record at all for that market on that day

The market roster for a district is defined as *every market that appears at least once in the
loaded data for that district* — the plan's documented fallback (Section 8, Risk register) for
when a canonical 240-340-market list isn't independently available. This is stated here explicitly
rather than silently assumed.

In [ ]:
def build_submission_status(df: pd.DataFrame, week_start, week_end):
    """Returns {district: {market: {date_str: status}}}."""
    if week_start is None:
        return {}
    all_days = [week_start + timedelta(days=i) for i in range((week_end - week_start).days + 1)]
    status = defaultdict(lambda: defaultdict(dict))

    roster = defaultdict(set)
    for _, row in df.iterrows():
        if row["district"]:
            roster[row["district"]].add(row["market"])

    has_trade = defaultdict(set)     # (district, market, date) with a usable modal price
    has_record = defaultdict(set)    # (district, market, date) with any record at all
    for _, row in df.iterrows():
        d, m, dt = row["district"], row["market"], row["arrival_date_parsed"]
        if not d or dt is None or dt not in all_days:
            continue
        has_record[(d, m)].add(dt)
        if pd.notna(row["modal_price"]):
            has_trade[(d, m)].add(dt)

    for district, markets in roster.items():
        for market in markets:
            for day in all_days:
                if day in has_trade[(district, market)]:
                    s = "submitted-with-trade"
                elif day in has_record[(district, market)]:
                    s = "submitted-no-transaction"
                else:
                    s = "not-submitted"
                status[district][market][str(day)] = s
    return status


submission_status = build_submission_status(current_week_df, week_start, week_end) if week_end else {}
print(f"Districts with a submission-status table: {len(submission_status)}")

## 7. Per-district aggregation

Computes, per district, everything in the Section 3/6 checklist that doesn't require the
week-on-week comparison (that's section 8):
- market detail + reporting-day counts, full-week (5+ day) reporters, market that reported most days
- top commodities by **arrival volume (tonnes)** when arrival_qty is present in the data, falling
  back to price-quote count with an explicit flag if tonnage wasn't available in the loaded files
  (per the plan: never silently substitute one ranking for the other)
- horticulture-category flag, driven by `commodity_group` when the source provides it

In [ ]:
HORTICULTURE_GROUP_KEYWORDS = ["fruit", "vegetable", "horticulture"]
HORTICULTURE_COMMODITY_FALLBACK = {
    "tomato", "onion", "potato", "brinjal", "cauliflower", "cabbage", "okra",
    "chilli", "green chilli", "capsicum", "carrot", "peas", "cucumber",
}


def is_horticulture(row):
    grp = str(row.get("commodity_group") or "").strip().lower()
    if grp and grp != "nan":
        return any(kw in grp for kw in HORTICULTURE_GROUP_KEYWORDS)
    cmdt = str(row.get("commodity") or "").strip().lower()
    return cmdt in HORTICULTURE_COMMODITY_FALLBACK


def aggregate_district(df: pd.DataFrame, district: str, week_start, week_end):
    d = df[df["district"] == district]
    if d.empty:
        return None

    market_rows = []
    for market, g in d.groupby("market"):
        reporting_days = g["arrival_date_parsed"].nunique()
        market_rows.append({
            "market": market,
            "reporting_days": int(reporting_days),
            "full_week_reporter": bool(reporting_days >= 5),
            "commodity_count": int(g["commodity"].nunique()),
        })
    market_rows.sort(key=lambda x: -x["reporting_days"])

    has_tonnage = d["arrival_qty"].notna().any()
    if has_tonnage:
        top_by_volume = (
            d.dropna(subset=["arrival_qty"])
             .groupby("commodity")["arrival_qty"].sum()
             .sort_values(ascending=False)
             .head(5)
        )
        top_commodities = [{"commodity": k, "arrival_qty_tonnes": round(v, 2)} for k, v in top_by_volume.items()]
        ranking_basis = "arrival_qty_tonnes"
    else:
        top_by_count = d["commodity"].value_counts().head(5)
        top_commodities = [{"commodity": k, "price_quote_count": int(v)} for k, v in top_by_count.items()]
        ranking_basis = "price_quote_count (arrival_qty not present in loaded data)"

    hort_mask = d.apply(is_horticulture, axis=1)
    horticulture_commodities = sorted(d.loc[hort_mask, "commodity"].dropna().unique().tolist())

    return {
        "district": district,
        "week_start": str(week_start),
        "week_end": str(week_end),
        "markets_reporting": len(market_rows),
        "markets_full_week_5plus_days": sum(1 for m in market_rows if m["full_week_reporter"]),
        "top_reporting_market": market_rows[0]["market"] if market_rows else None,
        "market_detail": market_rows,
        "top_commodities": top_commodities,
        "top_commodities_ranking_basis": ranking_basis,
        "horticulture_commodities_present": horticulture_commodities,
        "submission_status": submission_status.get(district, {}),
    }


district_aggregates = {}
if week_end:
    for district in sorted(current_week_df["district"].dropna().unique()):
        district_aggregates[district] = aggregate_district(current_week_df, district, week_start, week_end)

print(f"Districts aggregated: {len(district_aggregates)}")
list(district_aggregates.keys())[:10]

## 8. Week-on-week price change

Compares modal price per commodity+market between the current and prior week. Only commodities
present in **both** weeks are ranked — no interpolation for a commodity missing from either week.
Ranks the top 3 gainers and top 3 decliners by percent change, per district.

In [ ]:
def compute_price_deviation(current_df: pd.DataFrame, prior_df: pd.DataFrame, district: str):
    cur = (
        current_df[current_df["district"] == district]
        .dropna(subset=["modal_price"])
        .groupby("commodity")["modal_price"].mean()
    )
    prior = (
        prior_df[prior_df["district"] == district]
        .dropna(subset=["modal_price"])
        .groupby("commodity")["modal_price"].mean()
    )
    common = cur.index.intersection(prior.index)
    if len(common) == 0:
        return {"available": False, "reason": "no overlapping commodities between current and prior week", "top_gainers": [], "top_decliners": []}

    pct_change = ((cur[common] - prior[common]) / prior[common] * 100).round(2)
    ranked = pct_change.sort_values(ascending=False)
    top_gainers = [{"commodity": k, "pct_change": v} for k, v in ranked.head(3).items()]
    top_decliners = [{"commodity": k, "pct_change": v} for k, v in ranked.tail(3).sort_values().items()]
    return {"available": True, "top_gainers": top_gainers, "top_decliners": top_decliners}


if week_end and not prior_week_df.empty:
    for district, agg in district_aggregates.items():
        agg["price_change_vs_prior_week"] = compute_price_deviation(current_week_df, prior_week_df, district)
elif week_end:
    print("No prior-week data loaded — price_change_vs_prior_week left unavailable for every district "
          "(need a second week of CSV/PDF exports in data/csv or data/pdf).")
    for agg in district_aggregates.values():
        agg["price_change_vs_prior_week"] = {"available": False, "reason": "prior week not loaded", "top_gainers": [], "top_decliners": []}

## 9. Write fact sheets

One JSON file per district — the *only* object that should ever be handed to the narration model
in the later (Day 3) step. Nothing here is inferred by a model; everything is a raw field or a
value computed above.

In [ ]:
written = []
for district, fact_sheet in district_aggregates.items():
    safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", district)
    out_path = FACT_SHEET_DIR / f"{safe_name}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(fact_sheet, f, indent=2, ensure_ascii=False, default=str)
    written.append(out_path.name)

print(f"Wrote {len(written)} fact sheets to {FACT_SHEET_DIR}")
written[:10]

## 10. Spot-check (manual sanity check before anything downstream touches this)

Per the plan (Day 2 exit criteria): hand-verify one district's fact sheet against its raw records
before treating the pipeline as trustworthy. Change `SPOT_CHECK_DISTRICT` to any district present
in `district_aggregates`.

In [ ]:
SPOT_CHECK_DISTRICT = next(iter(district_aggregates), None)

if SPOT_CHECK_DISTRICT:
    print(f"Spot-checking: {SPOT_CHECK_DISTRICT}\n")
    print(json.dumps(district_aggregates[SPOT_CHECK_DISTRICT], indent=2, default=str))
    print("\nRaw rows backing this district (current week):")
    display(current_week_df[current_week_df["district"] == SPOT_CHECK_DISTRICT])
else:
    print("Nothing to spot-check yet — load CSV/PDF files into data/csv or data/pdf and re-run.")

## Not yet in this notebook (by design)

Per `final_implementation_plan_1.docx`, this notebook covers Day 1 (ingest) + Day 2 (compute).
Still open, deliberately not started here until Day 2's output above is verified against real
data:
- **Numeric diff-checker** (Day 2, Step 2.7) — regex-extract every number from generated brief
  text and hard-match against these fact sheets.
- **Qwen3 narration + IndicTrans2 translation** (Day 3) — reads a fact sheet JSON as its only
  input.
- **HHEM-2.1-Open grounding score** (Day 3) — secondary signal, not a gate.
- **Sampled human review + HTML/DOCX export** (Day 4).